# CS 691 · Module 13 · Reading event histories

Decide whether one request’s records support a required order of events, and explain how a missing or tied time limits the answer.

**Before you begin.** You have checked whether a proposed plan puts its steps in a permissible order. Now the desk has already acted. You will check what the records say happened.

A plan said “check ID, then hand over C1.” The desk wants to know what actually happened on Ava’s request. What would you look for?

<details><summary>Reveal the answer</summary>

Look for observations of each action tied to Ava’s request, with their event times. The plan alone gives the intended order.

</details>

## What happened to this request?

A plan lists intended steps. An event log records observations of steps that took place. Each row needs a case identifier so that events about the same request stay together. One borrower or one camera can appear in several cases.

Ava requests camera C1 twice. The desk calls her first request R7 and her second R8. In the code each row contains a request, an action and the minute after opening. The loop reads one row at a time.

**Predict before running the cell.** Could Ava’s name and the camera’s ID, together, replace the request ID in this example?

In [ ]:
events = [("R7", "check", 2), ("R8", "check", 7),
          ("R7", "handoff", 5)]
for request, action, minute in events:
    if request == "R7":
        print(minute, action)

<details><summary>Reveal output and explanation</summary>

```text
2 check
5 handoff
```

No. Both requests would share exactly those values. The request ID lets us determine which check belongs with which handoff.

</details>

## Seeing two events is different from ordering them

Our fictional desk requires an ID check before handing over C1. A list of action names tells us which actions appear. A time for each action lets us test the word “before.” We start with cases that contain one check and one handoff.

Two requests contain the same actions. For each request, compare the minute of the check with the minute of the handoff. Python’s less-than sign asks whether the first time is strictly earlier.

**Predict before running the cell.** R8 is marked False even though the log shows an ID check. What exactly failed?

In [ ]:
for request, check, handoff in [("R7", 2, 5), ("R8", 7, 4)]:
    print(request, check < handoff)

<details><summary>Reveal output and explanation</summary>

```text
R7 True
R8 False
```

R8’s check at minute 7 came after its handoff at minute 4. The presence of the check alone cannot establish the required order.

</details>

## Leave room for an incomplete record

A missing check entry can mean the check was skipped or that its record is missing. Equal timestamps can hide the actual order when the clock only counts whole minutes. Keep these two kinds of uncertainty separate from a recorded reversal.

Use None for an absent time. Test for missing values before comparing numbers. For this exercise the two timestamps use the same clock.

**Predict before running the cell.** What additional evidence would help when both events show minute 6?

In [ ]:
def order(check, handoff):
    if check is None or handoff is None:
        return "missing record"
    if check == handoff:
        return "order unknown"
    return "in order" if check < handoff else "reversed"

for pair in [(None, 6), (6, 6), (7, 6)]:
    print(order(*pair))

<details><summary>Reveal output and explanation</summary>

```text
missing record
order unknown
reversed
```

A reliable within-minute sequence number or more precise event times for the same request could establish which action occurred first.

</details>

## Explain the result using the original records

A clerk may enter a check after the handoff even when the check happened first. Retain both the event time and the entry time. An explanation should name the evidence rows, say which clock it used and describe only what those records support.

Rows E12 and E13 both concern Ava’s R7. The columns are evidence ID, action, event minute and entry minute. Sorting by each of the last two columns answers a different question.

**Predict before running the cell.** A clerk says, “The check came after handoff because E12 was entered last.” What correction can you support?

In [ ]:
rows = [("E12", "check", 2, 8), ("E13", "handoff", 5, 5)]
for column in [2, 3]:
    ordered = sorted(rows, key=lambda row: row[column])
    print(" then ".join(row[0] for row in ordered))

<details><summary>Reveal output and explanation</summary>

```text
E12 then E13
E13 then E12
```

E12 records the check at minute 2, before the minute-5 handoff in E13. Its later entry at minute 8 describes when the record arrived. These records support the required event order.

</details>

## Compare event time with entry time

Positions in a Python row start at zero. Sorting by column 2 arranges rows by event time, and sorting by column 3 arranges them by entry time. The program prints only the IDs so you can compare the two sequences directly.

Replace this line:
rows = [("E12", "check", 2, 8), ("E13", "handoff", 5, 5)]
with:
rows = [("E12", "check", 2, 8), ("E13", "handoff", 5, 9)]
What will print?

In [ ]:
rows = [("E12", "check", 2, 8), ("E13", "handoff", 5, 5)]
for column in [2, 3]:
    ordered = sorted(rows, key=lambda row: row[column])
    print(" then ".join(row[0] for row in ordered))

<details><summary>Worked answer after your change</summary>

E12 then E13
E12 then E13

Only the handoff’s entry time changed. With 8 before 9, the order of the entries now agrees with the event order.

</details>

## Exercise 1

Request R11 has handoff H1 at 12, check C2 at 14 and an earlier check C1 at 9. The rule requires at least one earlier check. Does H1 pass?

<details><summary>Hint</summary>

Compare each check with the same handoff. One qualifying pair is sufficient.

</details>

<details><summary>Worked answer</summary>

Yes. C1 at 9 is earlier than H1 at 12. C2 at 14 cannot support H1, but the rule only asks for one earlier check.

</details>

## Exercise 2

A return and a new loan share a camera. Which columns would let someone audit the order of steps in each loan even if the logs arrive late?

<details><summary>Hint</summary>

Consider identity, the action itself, both clocks and where to find the row again.

</details>

<details><summary>Worked answer</summary>

Keep a distinct loan ID, item ID, action, event time, entry time and evidence-row ID. Group by loan, order by event time and use row IDs to cite the evidence. Record clock precision or a reliable sequence number if times can tie.

</details>

## Exercise 3

A dashboard counts 7 in order, 2 reversed and 1 missing. Of the 10 cases, what fraction has evidence of following the rule? Of the cases whose order is known, what fraction followed it?

<details><summary>Hint</summary>

The first denominator is all cases. For the second, drop the unresolved case.

</details>

<details><summary>Worked answer</summary>

Seven of ten cases have supporting evidence. Among the nine with known order, seven of nine followed it. Report the missing case so readers know why the fractions differ.

</details>

Suppose a log shows that an extra clerk arrived before a queue cleared. Would the same queue have cleared just as quickly without that clerk? Module 14 gives that question a precise meaning.

The recorded wait fell just after an extra clerk arrived. What outcome is missing if you want to know whether the extra clerk caused the drop?

<details><summary>Reveal the answer</summary>

You would need to know how the queue would have changed over the same period without extra help. The event order establishes which happened first and leaves that alternative outcome unobserved.

</details>